# The PAV × pleiotropy interaction, tested formally

Notebook 05 showed the pleiotropy ceiling in stratified contrasts: it is strong among PAV-supported
pairs and weak without a PAV, in both resources. That is an interaction claim, but a pair of
stratum-specific P values is not a test of it. This notebook fits the interaction explicitly and
reports its P value in each resource.

## Specification

Restricted to **genetically supported** pairs, because the interaction is a statement about what kind
of support works, not about support versus none. Two binary factors:

- `pav` — the supporting credible set contains a protein-altering variant
- `low` — the target's therapeutic-area count is 2–5 (versus ≥ 6, high pleiotropy)

`outcome ~ pav + low + pav:low`. The interaction odds ratio is how much larger PAV's advantage is
among low-pleiotropy targets than among highly pleiotropic ones — equivalently, how much more the
pleiotropy ceiling matters when the support is protein-altering. Supported pairs with TA ≤ 1 are
excluded so `low` contrasts intermediate against high pleiotropy only.

Three P values per resource, because 9–23 successes per cell makes the choice of test matter:

- **Wald** — what a regression table prints, least reliable here;
- **likelihood ratio** — the interaction term against the additive model;
- **permutation** — `low` reshuffled among supported pairs within PAV strata, 10,000 times, so the
  null preserves both marginal effects and every cell size. This is the one to quote.

## What counts as replication

ChEMBL is where the interaction was found, so its P value is descriptive. The **Pharmaprojects P value
is the replication test**, and the direction was fixed in advance by ChEMBL, so a one-sided P value is
legitimate and is reported alongside the two-sided one.

49% of Pharmaprojects' launched pairs are also ChEMBL phase 4, so the two are not independent. The
last section therefore repeats the test on Pharmaprojects pairs **absent from ChEMBL altogether** — a
smaller, genuinely non-overlapping replication.

In [1]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

from or10_stats import support_mask

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)

path_to_intermediate_data_folder = "../../../data/intermediate_files/"
chembl = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_chembl_master-r1.parquet")
pp = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_pharmaprojects_master-r1.parquet")

for df in (chembl, pp):
    df["ta"] = df["uniqueTherapeuticAreas"].fillna(0).astype(float)
    df["gps"] = df["uniqueDiseases"].fillna(0).astype(float)
    df["support_all"] = support_mask(df).astype(int)
    df["support_pav"] = support_mask(df, pav=True).astype(int)

N_PERM = 10_000
SEED = 20260811

## The 2 × 2 of supported pairs

In [2]:
def interaction_frame(df, split_at=6, low_min=2, pleiotropy="ta"):
    """Supported pairs with a PAV flag and a low-versus-high pleiotropy flag."""
    sup = df[df["support_all"] == 1].copy()
    sup = sup[sup["in_gps"] & (sup[pleiotropy] >= low_min)]  # drop supported pairs below the floor
    sup["pav"] = sup["support_pav"].astype(int)
    sup["low"] = (sup[pleiotropy] < split_at).astype(int)
    return sup[["targetId", "diseaseId", "approved", "pav", "low"]].reset_index(drop=True)


def cell_table(sup, label):
    """Success counts and rates in the four cells."""
    g = sup.groupby(["pav", "low"])["approved"].agg(["sum", "size"]).reset_index()
    g["rate"] = g["sum"] / g["size"]
    g["cell"] = np.where(g["pav"] == 1, "PAV", "no PAV") + ", " + np.where(g["low"] == 1, "TA 2-5", "TA >=6")
    g["dataset"] = label
    return g[["dataset", "cell", "pav", "low", "sum", "size", "rate"]].rename(
        columns={"sum": "approved", "size": "pairs"}
    )


DATASETS = [("ChEMBL", chembl), ("Pharmaprojects", pp)]
frames = {label: interaction_frame(df) for label, df in DATASETS}
cells_table = pd.concat([cell_table(frames[label], label) for label, _ in DATASETS], ignore_index=True)
print(cells_table.round(4).to_string(index=False))
print()
for label in frames:
    g = frames[label].groupby(["pav", "low"])["approved"].agg(["sum", "size"])
    r = {k: v["sum"] / v["size"] for k, v in g.iterrows()}
    ratio_pav = (r[(1, 1)] / (1 - r[(1, 1)])) / (r[(1, 0)] / (1 - r[(1, 0)]))
    ratio_nonpav = (r[(0, 1)] / (1 - r[(0, 1)])) / (r[(0, 0)] / (1 - r[(0, 0)]))
    print(
        f"{label}: ceiling OR among PAV-supported {ratio_pav:.2f}, without PAV {ratio_nonpav:.2f}, "
        f"ratio of ratios {ratio_pav / ratio_nonpav:.2f}"
    )

       dataset           cell  pav  low  approved  pairs   rate
        ChEMBL no PAV, TA >=6    0    0        61    218 0.2798
        ChEMBL no PAV, TA 2-5    0    1        88    311 0.2830
        ChEMBL    PAV, TA >=6    1    0        20     67 0.2985
        ChEMBL    PAV, TA 2-5    1    1        51     87 0.5862
Pharmaprojects no PAV, TA >=6    0    0        30    164 0.1829
Pharmaprojects no PAV, TA 2-5    0    1        18    142 0.1268
Pharmaprojects    PAV, TA >=6    1    0         9     69 0.1304
Pharmaprojects    PAV, TA 2-5    1    1        23     60 0.3833

ChEMBL: ceiling OR among PAV-supported 3.33, without PAV 1.02, ratio of ratios 3.28
Pharmaprojects: ceiling OR among PAV-supported 4.14, without PAV 0.65, ratio of ratios 6.39


## The interaction test

In [3]:
def interaction_test(sup, label, n_perm=N_PERM, seed=SEED):
    """Interaction odds ratio with Wald, likelihood-ratio and permutation P values."""
    full = smf.logit("approved ~ pav + low + pav:low", data=sup).fit(disp=False)
    additive = smf.logit("approved ~ pav + low", data=sup).fit(disp=False)

    term = "pav:low"
    coef = float(full.params[term])
    ci = np.exp(full.conf_int().loc[term].values)
    lr = 2 * (float(full.llf) - float(additive.llf))

    # permute `low` within PAV strata: preserves both main effects and all four cell sizes
    rng = np.random.default_rng(seed)
    work = sup.reset_index(drop=True).copy()
    positions = [np.flatnonzero((work["pav"] == v).to_numpy()) for v in (0, 1)]
    base_low = work["low"].to_numpy()
    exceed_abs = exceed_signed = valid = 0
    sign = 1.0 if coef >= 0 else -1.0
    # thin permuted tables can separate; those fits are dropped and counted, and their convergence
    # warnings are silenced so they do not bury the output
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _ in range(n_perm):
            low_perm = base_low.copy()
            for pos in positions:
                low_perm[pos] = rng.permutation(low_perm[pos])
            work["low_perm"] = low_perm
            try:
                fit = smf.logit("approved ~ pav + low_perm + pav:low_perm", data=work).fit(disp=False)
                stat = float(fit.params["pav:low_perm"])
            except Exception:
                continue
            valid += 1
            if abs(stat) >= abs(coef):
                exceed_abs += 1
            if sign * stat >= sign * coef:
                exceed_signed += 1
    perm_p = (exceed_abs + 1) / (valid + 1)
    perm_p_one_sided = (exceed_signed + 1) / (valid + 1)

    return {
        "dataset": label,
        "interaction_or": float(np.exp(coef)),
        "ci_low": float(ci[0]),
        "ci_high": float(ci[1]),
        "p_wald": float(full.pvalues[term]),
        "p_lrt": float(chi2.sf(lr, 1)),
        "p_permutation_two_sided": perm_p,
        "p_permutation_one_sided": perm_p_one_sided,
        "n_pairs": len(sup),
        "n_approved": int(sup["approved"].sum()),
        "n_permutations_used": valid,
    }


results = pd.DataFrame([interaction_test(frames[label], label) for label, _ in DATASETS])
results.round(5)

,dataset,interaction_or,ci_low,ci_high,p_wald,p_lrt,p_permutation_two_sided,p_permutation_one_sided,n_pairs,n_approved,n_permutations_used
0,ChEMBL,3.27784,1.50668,7.13107,0.00276,0.00238,0.0018,0.0010,683,220,10000
1,Pharmaprojects,6.39147,2.17413,18.78951,0.00075,0.00050,0.0010,0.0007,435,80,10000


In [4]:
for _, r in results.iterrows():
    print(
        f"{r['dataset']:15s} interaction OR {r['interaction_or']:.2f} [{r['ci_low']:.2f}, {r['ci_high']:.2f}]  "
        f"Wald p = {r['p_wald']:.4g}  LRT p = {r['p_lrt']:.4g}  permutation p = {r['p_permutation_two_sided']:.4g} "
        f"({int(r['n_approved'])} successes in {int(r['n_pairs'])} supported pairs)"
    )
print()
pp_row = results[results["dataset"] == "Pharmaprojects"].iloc[0]
print(f"replication P value (Pharmaprojects, two-sided permutation): {pp_row['p_permutation_two_sided']:.4g}")
print(f"  one-sided, direction pre-specified by ChEMBL:              {pp_row['p_permutation_one_sided']:.4g}")

ChEMBL          interaction OR 3.28 [1.51, 7.13]  Wald p = 0.002757  LRT p = 0.002383  permutation p = 0.0018 (220 successes in 683 supported pairs)
Pharmaprojects  interaction OR 6.39 [2.17, 18.79]  Wald p = 0.0007475  LRT p = 0.0004952  permutation p = 0.0009999 (80 successes in 435 supported pairs)

replication P value (Pharmaprojects, two-sided permutation): 0.0009999
  one-sided, direction pre-specified by ChEMBL:              0.0006999


## Sensitivity to the cut point and to the pleiotropy measure

The split at 6 therapeutic areas comes from the published window, so it is itself a chosen threshold.
The interaction is refitted at every cut point, and in gPS instead of therapeutic areas. Permutation
is skipped in the sweep (10,000 refits per cell would dominate the runtime); the LRT P value is
reported, which was in close agreement above.

In [5]:
sweep_rows = []
for label, df in DATASETS:
    for measure, cuts in [("ta", range(3, 11)), ("gps", [3, 5, 8, 10, 15, 20])]:
        for cut in cuts:
            sup = interaction_frame(df, split_at=cut, low_min=2 if measure == "ta" else 1, pleiotropy=measure)
            counts = sup.groupby(["pav", "low"])["approved"].agg(["sum", "size"])
            if len(counts) < 4 or counts["sum"].min() == 0:
                sweep_rows.append(
                    {
                        "dataset": label,
                        "measure": measure,
                        "cut": cut,
                        "interaction_or": np.nan,
                        "p_lrt": np.nan,
                        "note": "a cell has no successes",
                    }
                )
                continue
            full = smf.logit("approved ~ pav + low + pav:low", data=sup).fit(disp=False)
            additive = smf.logit("approved ~ pav + low", data=sup).fit(disp=False)
            sweep_rows.append(
                {
                    "dataset": label,
                    "measure": measure,
                    "cut": cut,
                    "interaction_or": float(np.exp(full.params["pav:low"])),
                    "p_lrt": float(chi2.sf(2 * (float(full.llf) - float(additive.llf)), 1)),
                    "n_pairs": len(sup),
                    "min_cell_successes": int(counts["sum"].min()),
                    "note": "",
                }
            )
sweep = pd.DataFrame(sweep_rows)

for (label, measure), sub in sweep.groupby(["dataset", "measure"], sort=False):
    print(f"{label} / {measure}")
    print(
        sub[["cut", "interaction_or", "p_lrt", "n_pairs", "min_cell_successes", "note"]].round(4).to_string(index=False)
    )
    print()

ChEMBL / ta
 cut  interaction_or  p_lrt  n_pairs  min_cell_successes note
   3          2.4332 0.1710      683                  10     
   4          1.4718 0.4002      683                  18     
   5          2.1785 0.0421      683                  31     
   6          3.2778 0.0024      683                  20     
   7          2.5495 0.0262      683                  15     
   8          2.5592 0.0380      683                  11     
   9          3.1030 0.0200      683                   9     
  10          2.5729 0.0665      683                   8     

ChEMBL / gps
 cut  interaction_or  p_lrt  n_pairs  min_cell_successes note
   3          0.6166 0.4442      742                   6     
   5          0.8524 0.7267      742                  15     
   8          0.9547 0.9035      742                  29     
  10          1.1442 0.7192      742                  34     
  15          1.2530 0.5495      742                  28     
  20          1.1056 0.7987      742        

## A non-overlapping replication

447 of 911 launched Pharmaprojects pairs are also ChEMBL phase 4, so the two resources share about half
their successes. Repeating the test on Pharmaprojects pairs **absent from the ChEMBL table entirely**
removes that dependence. It also removes most of the data, so the count of successes per cell is
printed and the result is interpreted with that in mind rather than by its P value alone.

In [6]:
chembl_pairs = set(map(tuple, chembl[["targetId", "diseaseId"]].values))
pp_only = pp[[t not in chembl_pairs for t in map(tuple, pp[["targetId", "diseaseId"]].values)]].copy()
print(
    f"Pharmaprojects pairs not present in ChEMBL: {len(pp_only)} of {len(pp)} "
    f"({int(pp_only['approved'].sum())} launched of {int(pp['approved'].sum())})"
)

sup_only = interaction_frame(pp_only)
print()
print(cell_table(sup_only, "Pharmaprojects, ChEMBL pairs removed").round(4).to_string(index=False))

counts = sup_only.groupby(["pav", "low"])["approved"].agg(["sum", "size"])
if len(counts) == 4 and counts["sum"].min() > 0:
    non_overlap = interaction_test(sup_only, "Pharmaprojects, ChEMBL pairs removed")
    print()
    print(
        f"interaction OR {non_overlap['interaction_or']:.2f} "
        f"[{non_overlap['ci_low']:.2f}, {non_overlap['ci_high']:.2f}], "
        f"LRT p = {non_overlap['p_lrt']:.4g}, permutation p = {non_overlap['p_permutation_two_sided']:.4g}, "
        f"{non_overlap['n_approved']} successes in {non_overlap['n_pairs']} supported pairs"
    )
else:
    non_overlap = {
        "dataset": "Pharmaprojects, ChEMBL pairs removed",
        "interaction_or": np.nan,
        "note": "a cell has no successes; the interaction is not estimable on this subset",
    }
    print()
    print("a cell has no successes: the interaction is not estimable on the non-overlapping subset")
    print(counts.to_string())

Pharmaprojects pairs not present in ChEMBL: 4786 of 7390 (275 launched of 913)

                             dataset           cell  pav  low  approved  pairs   rate
Pharmaprojects, ChEMBL pairs removed no PAV, TA >=6    0    0         4     92 0.0435
Pharmaprojects, ChEMBL pairs removed no PAV, TA 2-5    0    1         3     76 0.0395
Pharmaprojects, ChEMBL pairs removed    PAV, TA >=6    1    0         3     40 0.0750
Pharmaprojects, ChEMBL pairs removed    PAV, TA 2-5    1    1         3     28 0.1071



interaction OR 1.64 [0.17, 15.85], LRT p = 0.6702, permutation p = 0.6688, 13 successes in 236 supported pairs


## Heterogeneity between the resources

Whether the interaction differs in size between ChEMBL and Pharmaprojects, by pooling the supported
pairs with a resource indicator and testing the three-way term. **This is not a valid independence
test** — half the successes are shared, so the two samples are correlated and the standard error is
too small. It is reported only to show that the two estimates are not far apart.

In [7]:
pooled = pd.concat(
    [
        frames["ChEMBL"].assign(resource=0),
        frames["Pharmaprojects"].assign(resource=1),
    ],
    ignore_index=True,
)

three_way = smf.logit("approved ~ pav * low * resource", data=pooled).fit(disp=False)
two_way = smf.logit("approved ~ pav * low + resource + pav:resource + low:resource", data=pooled).fit(disp=False)
lr = 2 * (float(three_way.llf) - float(two_way.llf))
print(
    f"three-way term pav:low:resource  OR {np.exp(three_way.params['pav:low:resource']):.3f}, "
    f"Wald p = {three_way.pvalues['pav:low:resource']:.3g}, LRT p = {chi2.sf(lr, 1):.3g}"
)
print("(the interaction is not detectably different between the two resources; see the caveat above)")

three-way term pav:low:resource  OR 1.950, Wald p = 0.325, LRT p = 0.322
(the interaction is not detectably different between the two resources; see the caveat above)


## Sensitivity — one row per Pharmaprojects target–indication pair

Pharmaprojects indications are MeSH terms mapped to EFO/MONDO through the disease index, and gene
symbols are mapped to Ensembl ids. Both mappings are one-to-many and were exploded, so 7,390 rows
cover only 6,577 distinct Pharmaprojects T–I pairs (`ti_uid`): 807 pairs appear 2–4 times.

Those repeats are **not** genuinely multiple indications. The clinical outcome never differs within a
repeated pair (0 of 807 groups); 422 of the 762 disease-expanded groups contain an ancestor/descendant
pair; and most of the remainder are one concept under two ontologies — `MONDO_0005299` "brain
ischemia" with `HP_0002637` "Cerebral ischemia", or `MP_0001845` "inflammation" with `GO_0006954`
"inflammatory response". A repeat therefore duplicates one trial outcome instead of contributing a
second one, which makes the standard errors mildly anticonservative.

Point estimates should barely move, because genetic support differs between the mapped copies in only
34 of 807 groups (10 of 807 for PAV support). Two checks:

- **collapsed** — one row per `ti_uid`, supported if any of its mapped pairs is supported;
- **1/k weighted** — each row down-weighted by the number of copies of its pair, which keeps the
  mapping ambiguity rather than resolving it.

In [8]:
import statsmodels.api as sm

from or10_stats import or_rs

pp_sens = pp.copy()
pp_sens["k"] = pp_sens.groupby("ti_uid")["ti_uid"].transform("size")

collapsed = (
    pp_sens.groupby("ti_uid")
    .agg(
        approved=("approved", "max"),
        support_all=("support_all", "max"),
        support_pav=("support_pav", "max"),
        ta=("ta", "max"),
        in_gps=("in_gps", "max"),
    )
    .reset_index()
)
print(f"rows as-is {len(pp_sens)} | collapsed to distinct T-I pairs {len(collapsed)}")
print(
    "non-disease ontologies among the mapped indications:",
    ", ".join(
        f"{pre.rstrip('_')} {int(pp_sens['diseaseId'].str.startswith(pre).sum())}"
        for pre in ["GO_", "MP_", "HP_", "Orphanet_"]
    ),
    "rows",
)


# OR of the published PAV + 2-5 TA definition on a Pharmaprojects table
def frozen_definition(df):
    mask = (df["support_pav"] == 1) & df["in_gps"].astype(bool) & df["ta"].between(2, 5)
    return or_rs(mask, df["approved"])


# interaction OR and LRT P value, optionally with frequency weights
def interaction_sensitivity(df, weight_col=None):
    sup = df[(df["support_all"] == 1) & df["in_gps"].astype(bool) & (df["ta"] >= 2)].copy()
    sup["pav"] = sup["support_pav"].astype(int)
    sup["low"] = (sup["ta"] < 6).astype(int)
    if weight_col is None:
        full = smf.logit("approved ~ pav + low + pav:low", data=sup).fit(disp=False)
        additive = smf.logit("approved ~ pav + low", data=sup).fit(disp=False)
    else:
        w = 1.0 / sup[weight_col]
        binom = sm.families.Binomial()
        full = smf.glm("approved ~ pav + low + pav:low", data=sup, family=binom, freq_weights=w).fit()
        additive = smf.glm("approved ~ pav + low", data=sup, family=binom, freq_weights=w).fit()
    ci = np.exp(full.conf_int().loc["pav:low"].values)
    return {
        "interaction_or": float(np.exp(full.params["pav:low"])),
        "ci_low": float(ci[0]),
        "ci_high": float(ci[1]),
        "p_lrt": float(chi2.sf(2 * (float(full.llf) - float(additive.llf)), 1)),
        "n_pairs": len(sup),
        "n_approved": int(sup["approved"].sum()),
    }


sens_rows = []
for label, df, w in [
    ("as-is (primary)", pp_sens, None),
    ("collapsed", collapsed, None),
    ("1/k weighted", pp_sens, "k"),
]:
    fz = frozen_definition(df)
    sens_rows.append(
        {
            "variant": label,
            "frozen_or": fz["odds_ratio"],
            "frozen_ci_low": fz["ci_low"],
            "frozen_ci_high": fz["ci_high"],
            "frozen_n_support": fz["n_support"],
            "frozen_n_approved": fz["yes_evid-high_clinphase"],
            **interaction_sensitivity(df, w),
        }
    )
sensitivity = pd.DataFrame(sens_rows)
print()
print(sensitivity.round(4).to_string(index=False))

rows as-is 7390 | collapsed to distinct T-I pairs 6577
non-disease ontologies among the mapped indications: GO 101, MP 77, HP 141, Orphanet 39 rows

        variant  frozen_or  frozen_ci_low  frozen_ci_high  frozen_n_support  frozen_n_approved  interaction_or  ci_low  ci_high  p_lrt  n_pairs  n_approved
as-is (primary)     4.4980         2.6605          7.6047                60                 23          6.3915  2.1741  18.7895 0.0005      435          80
      collapsed     4.6273         2.6929          7.9513                56                 22          6.6819  2.1848  20.4355 0.0006      406          77
   1/k weighted     4.4980         2.6605          7.6047                60                 23          6.4631  2.0610  20.2673 0.0009      435          80


In [9]:
base = sensitivity.iloc[0]
for _, r in sensitivity.iterrows():
    print(
        f"{r['variant']:16s} frozen OR {r['frozen_or']:.2f} "
        f"[{r['frozen_ci_low']:.2f}, {r['frozen_ci_high']:.2f}]  |  "
        f"interaction OR {r['interaction_or']:.2f} [{r['ci_low']:.2f}, {r['ci_high']:.2f}], "
        f"LRT p = {r['p_lrt']:.4g}"
    )
print()
print(
    "No conclusion changes, so the exploded table stays as the primary analysis: it is what the "
    "published Pharmaprojects processing produced, and collapsing moves the interaction from "
    f"{base['interaction_or']:.2f} to {sensitivity.iloc[1]['interaction_or']:.2f}, p from "
    f"{base['p_lrt']:.1e} to {sensitivity.iloc[1]['p_lrt']:.1e}."
)

as-is (primary)  frozen OR 4.50 [2.66, 7.60]  |  interaction OR 6.39 [2.17, 18.79], LRT p = 0.0004952
collapsed        frozen OR 4.63 [2.69, 7.95]  |  interaction OR 6.68 [2.18, 20.44], LRT p = 0.0005567
1/k weighted     frozen OR 4.50 [2.66, 7.60]  |  interaction OR 6.46 [2.06, 20.27], LRT p = 0.0009086

No conclusion changes, so the exploded table stays as the primary analysis: it is what the published Pharmaprojects processing produced, and collapsing moves the interaction from 6.39 to 6.68, p from 5.0e-04 to 5.6e-04.


## Export

In [10]:
cells_table.to_csv(path_to_intermediate_data_folder + "or10_interaction_cells-r1.csv", index=False)
results.to_csv(path_to_intermediate_data_folder + "or10_interaction_test-r1.csv", index=False)
sweep.to_csv(path_to_intermediate_data_folder + "or10_interaction_sweep-r1.csv", index=False)
sensitivity.to_csv(path_to_intermediate_data_folder + "or10_interaction_pp_duplication_sensitivity-r1.csv", index=False)
pd.DataFrame([non_overlap]).to_csv(
    path_to_intermediate_data_folder + "or10_interaction_non_overlap-r1.csv", index=False
)
print("exported 5 tables")

exported 5 tables
